# ddharmon Tutorial: Mapping Biological Entities with BioMapper2

This notebook demonstrates how to use the **ddharmon** package to resolve biological entity names to standardized identifiers using the BioMapper2 API.

## What is BioMapper2?

BioMapper2 is a knowledge graph-backed service that maps human-readable names (like "Glucose" or "BRCA1") to standardized **CURIEs** (Compact URIs). These CURIEs provide unambiguous identifiers that work across databases.

### Understanding CURIEs

A **CURIE** looks like `CHEBI:17234` or `HMDB:HMDB0000122`. It has two parts:
- **Prefix**: The namespace/database (e.g., `CHEBI`, `HMDB`, `UniProtKB`)
- **Local ID**: The identifier within that database

CURIEs solve the "what do you mean by glucose?" problem - instead of ambiguous names, you get precise database references.

### What ddharmon provides

- **Single lookups**: `map_entity("L-Histidine")` → `MappingResult`
- **Batch processing**: `map_entities([...])` with progress bars and rate limiting
- **Confidence scoring**: Know how reliable each mapping is
- **Multi-vocabulary results**: Get CHEBI, HMDB, RefMet, UniProt IDs in one call

---

## Part 1: Setup & Health Check

Let's start by importing ddharmon and verifying the API connection.

In [ ]:
# Load environment variables from .env file (if present)
from dotenv import load_dotenv
load_dotenv()  # Loads BIOMAPPER_API_KEY from .env file

# Required for running async code in Jupyter notebooks
import nest_asyncio
nest_asyncio.apply()

# Core imports
import ddharmon
from ddharmon import (
    map_entity,
    map_entities,
    summarize,
    BioMapperClient,
    MappingResult,
    MappingSummary,
    BioMapperError,
    BioMapperAuthError,
    BioMapperRateLimitError,
)

print(f"ddharmon version: {ddharmon.__version__}")

### API Key Configuration

ddharmon requires a `BIOMAPPER_API_KEY` environment variable. You can set it in several ways:

1. **Environment variable**: `export BIOMAPPER_API_KEY=your_key`
2. **`.env` file**: Create a file with `BIOMAPPER_API_KEY=your_key`
3. **In notebook** (not recommended for shared notebooks):
   ```python
   import os
   os.environ["BIOMAPPER_API_KEY"] = "your_key"
   ```

To request an API key, contact the BioMapper2 team or check your organization's credentials.

### Health Check

Before making mapping requests, let's verify the API is accessible. The `BioMapperClient` provides an async interface with a `health_check()` method.

In [ ]:
import asyncio

async def check_health():
    """Verify API connectivity and authentication."""
    async with BioMapperClient() as client:
        return await client.health_check()

health = asyncio.get_event_loop().run_until_complete(check_health())
print(f"API Health: {health}")

If you see `True` or a success response, you're ready to go! If you get an authentication error, check your `BIOMAPPER_API_KEY`.

---

## Part 2: Single Entity Lookup

The simplest operation is mapping a single entity name. Let's look up "L-Histidine", an amino acid.

### Using `map_entity()`

The `map_entity()` function takes a name string and returns a `MappingResult` object with all the mapping details.

In [ ]:
result = map_entity("L-Histidine")

print("=== MappingResult Fields ===")
print(f"Query Name:       {result.query_name}")
print(f"Resolved:         {result.resolved}")
print(f"Primary CURIE:    {result.primary_curie}")
print(f"Chosen KG ID:     {result.chosen_kg_id}")
print(f"Confidence Score: {result.confidence_score}")
print(f"Confidence Tier:  {result.confidence_tier}")
print()
print("=== Vocabulary-Specific IDs ===")
print(f"CHEBI IDs:        {result.ids_for('CHEBI')}")
print(f"HMDB IDs:         {result.ids_for('HMDB')}")
print(f"RefMet IDs:       {result.ids_for('refmet_id')}")

### Understanding the MappingResult

| Field | Description |
|-------|-------------|
| `query_name` | The original name you searched for |
| `resolved` | Boolean - did BioMapper2 find a match? |
| `primary_curie` | The best/canonical CURIE identifier |
| `chosen_kg_id` | Internal knowledge graph node ID |
| `confidence_score` | Numeric confidence (higher = better) |
| `confidence_tier` | Human-readable tier: high/medium/low/unknown |
| `identifiers` | Dictionary of all mapped vocabularies |

### Confidence Tiers

The `confidence_tier` property categorizes the numeric score:

- **high** (≥2.0): Strong match, use with confidence
- **medium** (1.0-2.0): Good match, may want to verify
- **low** (<1.0): Weak match, manual review recommended
- **unknown**: No confidence score available

### Async Client Usage

For more control (or when integrating into async code), use the `BioMapperClient` directly:

In [ ]:
async def map_with_client():
    """Demonstrate async client usage."""
    async with BioMapperClient() as client:
        result = await client.map_entity("Glucose")
        return result

async_result = asyncio.get_event_loop().run_until_complete(map_with_client())
print(f"Glucose -> {async_result.primary_curie} (confidence: {async_result.confidence_tier})")

---

## Part 3: Batch Metabolite Mapping

Real-world metabolomics data typically involves hundreds or thousands of compounds. The `map_entities()` function handles batches efficiently with built-in rate limiting.

### Toy Metabolite Dataset

Let's create a small dataset that represents typical metabolomics challenges:
- Well-known compound names
- Names with HMDB identifier hints
- Obscure or ambiguous names
- Vendor codes that won't resolve

In [ ]:
metabolite_records = [
    # Clean, well-known names (should resolve with high confidence)
    {"name": "Glucose"},
    {"name": "L-Carnitine"},
    {"name": "Sphinganine"},
    {"name": "Choline"},
    {"name": "Creatinine"},
    {"name": "Taurine"},
    {"name": "Uric acid"},
    
    # With HMDB hint (improves resolution)
    {"name": "Histidine", "identifiers": {"HMDB": "HMDB0000177"}},
    
    # Obscure names (medium/low confidence expected)
    {"name": "2-Hydroxyglutaric acid"},
    {"name": "3-Methylhistidine"},
    {"name": "Dimethylglycine"},
    
    # Vendor codes that won't resolve (expected failures)
    {"name": "Z1234567"},
    {"name": "AKOS016382737"},
    {"name": "UNKNOWN_COMPOUND_X"},
    {"name": "QC_STANDARD_001"},
]

print(f"Dataset: {len(metabolite_records)} compounds")

### Running the Batch

The `map_entities()` function processes records sequentially with a configurable delay between requests (default: 0.3s). Set `progress=True` to see a progress bar.

In [ ]:
# Map all metabolites with progress bar
results = map_entities(metabolite_records, progress=True)

print(f"\nMapped {len(results)} compounds")

### Summarizing Results

The `summarize()` function provides aggregate statistics about your batch mapping.

In [ ]:
summary = summarize(results)

print("=== Mapping Summary ===")
print(f"Total queries:     {summary.total_queried}")
print(f"Resolved:          {summary.resolved} ({summary.resolution_rate:.1%})")
print(f"Unresolved:        {summary.unresolved}")
print(f"Errors:            {summary.errors}")
print()
print("=== Vocabulary Coverage ===")
for vocab, count in sorted(summary.vocabulary_coverage.items()):
    print(f"  {vocab}: {count}")

### Results Table

Let's view all results in a tabular format:

In [ ]:
print(f"{'Query Name':<25} {'Primary CURIE':<25} {'Tier':<10}")
print("=" * 60)

for r in results:
    curie = r.primary_curie or "(not resolved)"
    tier = r.confidence_tier if r.resolved else "—"
    print(f"{r.query_name:<25} {curie:<25} {tier:<10}")

---

## Part 4: Gene/Protein Mapping

ddharmon can also map gene and protein names. The key difference is specifying `entity_type="biolink:Gene"` instead of the default `"biolink:SmallMolecule"`.

### Toy Gene Dataset

In [ ]:
gene_records = [
    {"name": "BRCA1"},   # Breast cancer susceptibility gene
    {"name": "TP53"},    # Tumor suppressor
    {"name": "EGFR"},    # Epidermal growth factor receptor
    {"name": "TNF"},     # Tumor necrosis factor
    {"name": "IL6"},     # Interleukin 6
    {"name": "APOE"},    # Apolipoprotein E
    {"name": "ACE2"},    # Angiotensin-converting enzyme 2
    {"name": "PCSK9"},   # Proprotein convertase
]

print(f"Dataset: {len(gene_records)} genes")

### Mapping with entity_type

The `entity_type` parameter tells BioMapper2 how to route the query. For genes/proteins, use `"biolink:Gene"` or `"biolink:Protein"`.

In [ ]:
# Map genes (note the entity_type parameter)
gene_results = map_entities(
    gene_records,
    entity_type="biolink:Gene",
    progress=True
)

# Show summary
gene_summary = summarize(gene_results)
print(f"\nResolved: {gene_summary.resolved}/{gene_summary.total_queried}")
print(f"\nVocabulary coverage: {gene_summary.vocabulary_coverage}")

### Comparing Vocabulary Coverage

Notice that gene mappings return different vocabularies than metabolites:
- **Metabolites**: CHEBI, HMDB, RefMet, PubChem, etc.
- **Genes**: HGNC, UniProtKB, NCBI Gene, Ensembl, etc.

In [ ]:
print(f"{'Gene':<10} {'Primary CURIE':<25} {'UniProt':<20}")
print("=" * 55)

for r in gene_results:
    curie = r.primary_curie or "(not resolved)"
    uniprot = r.ids_for("UniProtKB") or r.ids_for("uniprot") or []
    uniprot_str = ", ".join(uniprot[:2]) if uniprot else "—"
    print(f"{r.query_name:<10} {curie:<25} {uniprot_str:<20}")

---

## Part 5: Error Handling & Edge Cases

Production code should handle potential errors gracefully. ddharmon provides specific exception classes for different failure modes.

### Exception Classes

| Exception | When it's raised |
|-----------|------------------|
| `BioMapperError` | Base class for all ddharmon errors |
| `BioMapperAuthError` | Invalid or missing API key |
| `BioMapperRateLimitError` | Too many requests (429 response) |
| `BioMapperTimeoutError` | Request timed out |
| `BioMapperServerError` | Server-side error (5xx response) |

### Recommended Error Handling Pattern

In [ ]:
from ddharmon import (
    BioMapperError,
    BioMapperAuthError,
    BioMapperRateLimitError,
    BioMapperTimeoutError,
)

def safe_map_entity(name: str) -> MappingResult | None:
    """Map an entity with proper error handling."""
    try:
        return map_entity(name)
    except BioMapperAuthError:
        print(f"Authentication failed - check BIOMAPPER_API_KEY")
        return None
    except BioMapperRateLimitError as e:
        print(f"Rate limited! Retry after {e.retry_after}s")
        return None
    except BioMapperTimeoutError:
        print(f"Request timed out for '{name}'")
        return None
    except BioMapperError as e:
        print(f"Mapping error for '{name}': {e}")
        return None

# Test with a valid query
result = safe_map_entity("Caffeine")
if result:
    print(f"Success: {result.query_name} -> {result.primary_curie}")

### Unresolved Results

When BioMapper2 can't find a match, you still get a `MappingResult` - but `resolved=False`. Let's look at one of our unresolved vendor codes:

In [ ]:
# Find an unresolved result from our batch
unresolved = [r for r in results if not r.resolved]

if unresolved:
    example = unresolved[0]
    print(f"Query Name:       {example.query_name}")
    print(f"Resolved:         {example.resolved}")
    print(f"Primary CURIE:    {example.primary_curie}")
    print(f"Confidence Score: {example.confidence_score}")
    print(f"Confidence Tier:  {example.confidence_tier}")
    print(f"\nAll unresolved: {[r.query_name for r in unresolved]}")
else:
    print("All entities resolved!")

### Common Gotchas

**1. Rate Limiting**

If you're getting 429 errors, increase the delay between requests:
```python
results = map_entities(records, rate_limit_delay=0.5)  # 0.5s instead of default 0.3s
```

**2. Vendor Codes Won't Resolve**

Internal codes like `Z1234567` or `QC_STANDARD_001` are not in public knowledge graphs. This is expected behavior - you'll need to maintain your own mapping for these.

**3. Entity Type Matters**

Searching for "ACE2" as a `SmallMolecule` won't find the gene. Always use the appropriate `entity_type`:
- Metabolites: `"biolink:SmallMolecule"` (default)
- Genes: `"biolink:Gene"`
- Proteins: `"biolink:Protein"`

**4. HMDB Hints Improve Resolution**

If you have partial identifiers, include them:
```python
{"name": "Histidine", "identifiers": {"HMDB": "HMDB0000177"}}
```

---

## Part 6: Working with Results

Once you have mapping results, you'll often need to filter, extract, and analyze them.

### Filtering by Confidence

Separate high-confidence results from those needing review:

In [ ]:
# Filter by confidence tier
high_confidence = [r for r in results if r.confidence_tier == "high"]
needs_review = [r for r in results if r.confidence_tier in ("medium", "low")]
unresolved = [r for r in results if not r.resolved]

print(f"High confidence:  {len(high_confidence)} compounds")
print(f"Needs review:     {len(needs_review)} compounds")
print(f"Unresolved:       {len(unresolved)} compounds")

if needs_review:
    print(f"\nCompounds to review: {[r.query_name for r in needs_review]}")

### Extracting Vocabulary-Specific IDs

Often you need IDs for a specific database (e.g., CHEBI for pathway tools, HMDB for metabolomics databases):

In [ ]:
# Build a mapping: compound name -> CHEBI IDs
chebi_mapping = {
    r.query_name: r.ids_for("CHEBI") 
    for r in results 
    if r.resolved and r.ids_for("CHEBI")
}

print("CHEBI ID Mapping:")
for name, chebi_ids in list(chebi_mapping.items())[:5]:
    print(f"  {name}: {chebi_ids}")

### Vocabulary Coverage Analysis

The summary's `vocabulary_coverage` shows which databases had identifiers:

In [ ]:
# Get summary for resolved results only
resolved_results = [r for r in results if r.resolved]
resolved_summary = summarize(resolved_results)

print(f"Vocabulary coverage across {resolved_summary.total_queried} resolved compounds:")
print()

# Sort by count
for vocab, count in sorted(
    resolved_summary.vocabulary_coverage.items(),
    key=lambda x: x[1],
    reverse=True
):
    pct = count / resolved_summary.total_queried * 100
    bar = "█" * int(pct // 5)
    print(f"  {vocab:<15} {count:>3} ({pct:5.1f}%) {bar}")

---

## Next Steps

You've learned the core ddharmon API! Here are some next steps:

### For Metabolomics Workflows

If you're working with Metabolon data, check out the `ddharmon[metabolon]` extras:
```bash
pip install "ddharmon[metabolon]"
```

This adds preprocessing utilities for Metabolon Excel files. See the [biovector-eval](https://github.com/your-org/biovector-eval) project for examples.

### API Reference

**Core Functions:**
- `map_entity(name, identifiers=None, entity_type="biolink:SmallMolecule")` → `MappingResult`
- `map_entities(records, rate_limit_delay=0.3, entity_type="...", progress=False)` → `list[MappingResult]`
- `summarize(results)` → `MappingSummary`

**Async Client:**
```python
async with BioMapperClient() as client:
    await client.health_check()
    await client.map_entity("Glucose")
    await client.map_entities([...])
```

**MappingResult Properties:**
- `.query_name`, `.resolved`, `.primary_curie`, `.chosen_kg_id`
- `.confidence_score`, `.confidence_tier`
- `.identifiers` (dict), `.ids_for(vocab)` (method)

**MappingSummary Properties:**
- `.total`, `.resolved`, `.unresolved`, `.resolution_rate`
- `.vocabulary_coverage` (dict)